# Model-comparison evaluation

`DEFAULT_RUNS_ROOT` is the conventional processed-data `TASK/runs` directory. Each entry independently selects a `runs_root`, current directory leaf `run_name`, and optional label, so moved collections and Optuna trials can use different parents. Folder names remain storage aliases, scientific identities are read from the saved bundles.

Valid existing artifacts are reused without model reconstruction or inference. Missing roles are generated locally on explicit CPU by default. Incompatible, corrupt, stale, or partial targets are never rebuilt unless the visible rebuild option is deliberately enabled. Provisional terminal runs remain eligible when comparison science matches and are visibly labelled. Local notebook generation never contacts W&B.

The authoritative `normalized_group_macro_rmse` is dimensionless, lower is better, and is not a percentage.

In [ ]:
from pathlib import Path

from IPython.display import display as show

from src import analysis, common

evaluation_workflow = analysis.evaluation.workflow

TASK = "steady_flow"
DEFAULT_RUNS_ROOT: Path = common.paths.resolve_runs_root(TASK)

# Each run_name is a current directory leaf/storage alias, not the saved scientific run name.
RUNS = [
    {"runs_root": DEFAULT_RUNS_ROOT, "run_name": "replace_with_run_directory_a", "label": "Run A"},
    {"runs_root": DEFAULT_RUNS_ROOT, "run_name": "replace_with_run_directory_b", "label": "Run B"},
    {"runs_root": DEFAULT_RUNS_ROOT, "run_name": "replace_with_run_directory_c", "label": "Run C"},
    {"runs_root": DEFAULT_RUNS_ROOT, "run_name": "replace_with_run_directory_d", "label": "Run D"},
]

AUTO_BUILD_MISSING_ARTIFACTS = True
ARTIFACT_DEVICE = "cpu"
REBUILD_INCOMPATIBLE_ARTIFACTS = False
ARTIFACT_ROLES = ("id", "ood")

In [ ]:
run_selections = tuple(
    evaluation_workflow.EvaluationRunSelection(
        run_dir=selection["runs_root"] / selection["run_name"],
        label=selection["label"],
    )
    for selection in RUNS
)
context_specs = (
    evaluation_workflow.EvaluationContextSpec(key="validation_dataset", label="ID", artifact_role="id"),
    evaluation_workflow.EvaluationContextSpec(key="shifted_dataset", label="OOD", artifact_role="ood"),
)
evaluation = evaluation_workflow.prepare_evaluation_workflow(
    run_selections,
    context_specs,
    artifact_roles=ARTIFACT_ROLES,
    auto_build_missing=AUTO_BUILD_MISSING_ARTIFACTS,
    rebuild_incompatible=REBUILD_INCOMPATIBLE_ARTIFACTS,
    device_policy=ARTIFACT_DEVICE,
)
print(list(evaluation.report))

In [ ]:
show(evaluation.panel)